In [1]:
import numpy as np

from dftpy.constants import ENERGY_CONV
from edftpy import io
from edftpy.functional import LocalPP, KEDF, Hartree, XC
from edftpy.optimizer import Optimization
from edftpy.evaluator import EmbedEvaluator, TotalEvaluator
from edftpy.subsystem.subcell import SubCell, GlobalCell
from edftpy.interface import init_graphtopo
from edftpy.mpi import MP, sprint
from edftpy.engine.driver import DriverKS
from edftpy.engine.engine_qe import EngineQE
from edftpy.utils.common import Field, Functional, AbsFunctional
import multiprocessing

# Subsystem DFT simulation using eDFTpy

## A molecular system (e.g., water dimer) is partitioned into subsystems, each solved with its own Kohn–Sham (KS) engine (Quantum ESPRESSO). 

### A global embedding framework couples the subsystems through shared densities, external potentials, and an embedding functional. 
### The workflow constructs:

1. The global cell is built taking as input the coordinates of the entire system (GlobalCell).
2. Multiple subsystem KS drivers (DriverKS).
3. A total energy evaluator (TotalEvaluator)
4. An embedding evaluator (EmbedEvaluator).
5. and an optimization loop (Optimization) that self-consistently converges subsystem densities.

In [2]:
DATA = '../../../DATA/'

In [3]:
def get_optimizer(pplist,cellfile, subkeys, indices, ecut, cellsplit):
    '''
    Initialize Optimization object per subsystem
    
    (1) Reads atomic structure from file
    (2) Initializes parallel graph topology
    (3) Constructs global electronic system (GlobalCell) as gsystem
    (4) Builds subsystem KS drivers
    (5) Registers them into an Optimization object
    
    Returns the optimizer object
    '''
    ions = io.ase_read(cellfile)
    graphtopo = get_graphtopo([1,]*len(subkeys), parallel = True)
    gsystem = get_gsystem(ions, graphtopo, pplist, ecut)
    drivers = []
    for i, keysys in enumerate(subkeys):
        if graphtopo.isub != i and graphtopo.is_mpi:
            driver = None
        else :
            index = indices[i]
            driver = get_driver(keysys, ions, gsystem.grid, pplist, index, cellsplit, graphtopo)
        drivers.append(driver)

    graphtopo.build_region(grid=gsystem.grid, drivers=drivers)
    opt = Optimization(drivers = drivers, gsystem = gsystem, options={'econv': 1E-6*ions.nat})
    return opt

In [4]:
def get_gsystem(ions, graphtopo, pplist, ecut):
    '''
    Built Global System == Gsystm
    
    (1) Wraps MPI parallelization (MP)
    (2) Builds GlobalCell 
    (3) Attaches a total energy evaluator (TotalEvaluator)
    
    Returns the fully initialized global system
    '''
    
    mp_global = MP(comm = graphtopo.comm, parallel = graphtopo.is_mpi, decomposition = graphtopo.decomposition)
    gsystem = GlobalCell(ions, ecut = ecut, mp = mp_global, graphtopo = graphtopo)
    total_evaluator = get_total_evaluator(ions, gsystem.grid, pplist)
    gsystem.total_evaluator = total_evaluator
    return gsystem

In [5]:
def get_graphtopo(nprocs, parallel = False):
    '''
    (1) Creates graph-based partitioning object
    (2) Distributes processors across subsystems
    Returns topology manager used for parallel execution
    '''
    graphtopo = init_graphtopo(parallel)
    graphtopo.distribute_procs(nprocs)
    return graphtopo

In [6]:
def ext_functional(density,**kwargs):
    '''
    Defines a custom external energy functional (EXT0):

    (1) Computes a Thomas–Fermi-like density-dependent potential
    (2) Evaluates corresponding energy density integral
    
    Returns a Functional object containing: (1) energy and (2) potential
    '''
    factor = (3.0 / 10.0) * (5.0 / 3.0) * (3.0 * np.pi ** 2) ** (2.0 / 3.0)
    potential = factor * np.cbrt(density* density)
    energy=(potential*density).sum()*density.grid.dV*3.0/5.0
    obj = Functional(name = 'EXT0', energy=energy, potential=potential)
    return obj

In [7]:
class ExtFunctional(object):
    ''' Class version of an external potential functional (EXT1):

    (1) Stores a fixed external potential field vext
    (2) Computes energy as ⟨vext, ρ⟩
    
    Returns a Functional object
    '''
    def __init__(self, vext=None, **kwargs):
        self.vext=vext
        
    def __call__(self, density, **kwargs):
        potential=self.vext
        energy=(potential*density).sum()*density.grid.dV
        obj = Functional(name = 'EXT1', energy=energy, potential=potential)
        return obj

In [8]:
def get_total_evaluator(ions, grid, pplist):
    ''' Constructs the total energy functional evaluator for the global system:

    (1) Pseudopotential term (LocalPP)
    (2) Hartree energy
    (3) Exchange-correlation (PBE)
    (4) Kinetic energy functional (GGA / revAPBEK)
    (5) External functional (ext_functional)

    Return Total Evaluator
    '''
    xc_kwargs = {'xc' : 'PBE'}
    ke_kwargs = {'kedf' : 'GGA', 'k_str' : 'revAPBEK'}
    pseudo = LocalPP(grid = grid, ions=ions, PP_list=pplist)
    hartree = Hartree()
    xc = XC(**xc_kwargs)
    ke = KEDF(**ke_kwargs)
    funcdicts = {'XC' :xc, 'HARTREE' :hartree, 'PSEUDO' :pseudo, 'KE' :ke, 'EXT0': ext_functional}
    total_evaluator = TotalEvaluator(**funcdicts)
    return total_evaluator

In [9]:
def get_embed_evaluator(subcell):
    ''' Builds the embedding-specific energy evaluator for a subsystem:

    (1) XC + kinetic functionals
    (2) external potentials:
    (3) EXT0: density-dependent functional
    (4) EXT1: fixed field potential (ExtFunctional)
    
    returns EmbedEvaluator
    '''
    xc_kwargs = {'xc' : 'PBE'}
    ke_kwargs = {'kedf' : 'GGA', 'k_str' : 'revAPBEK'}
    xc = XC(**xc_kwargs)
    ke = KEDF(**ke_kwargs)
    # External Potential--------------------------------------------
    vext = Field(grid=subcell.grid)
    vext[:]= -1E-6
    extobj = ExtFunctional(vext)
    #---------------------------------------------------------------
    emb_funcdicts = {'XC' :xc, 'KE' :ke, 'EXT0': ext_functional, 'EXT1': extobj}
    embed_evaluator = EmbedEvaluator(**emb_funcdicts)
    return embed_evaluator

In [10]:
def get_driver(prefix, ions, grid, pplist, index, cellsplit, graphtopo):
    ''' Creates a subsystem KS driver (Quantum ESPRESSO engine):

    (1) Constructs SubCell (subsystem region of global system)
    (2) Initializes subsystem density
    (3) Builds embedding evaluator
    (4) Sets QE parameters (ecut, pseudopotentials, mixing)
    
    Returns DriverKS
    '''
    mp = MP(comm = graphtopo.comm_sub, decomposition = graphtopo.decomposition)
    subcell = SubCell(ions, grid, index = index, cellsplit = cellsplit, mp = mp)
    # given a negative value which means will get from driver
    subcell.density[:] = -1.0
    embed_evaluator = get_embed_evaluator(subcell)
    cell_params = {'pseudopotentials' : pplist}
    params = {'system' : {'ecutwfc' : 600*ENERGY_CONV["eV"]["Hartree"]*2}}
    margs= {
            'evaluator' : embed_evaluator,
            'prefix' : prefix,
            'subcell' : subcell,
            'cell_params': cell_params,
            'params': params,
            'exttype' : 3, # 3 is XC embedded, 7 is without XC
            'mixer' : 0.7
            }
    engine = EngineQE()
    driver = DriverKS(engine = engine, **margs)
    return driver

## It then runs a SCF-like optimization and periodically prints:

1. The charge of the total system.
2. The charge per subsystem.
3. The contribution of the external potential.

### 1. Define the subsystem cell and partition

In [11]:
# cellfile = f'{DATA}/h2o_2.xyz'
# subkeys = ['sub_ks_0', 'sub_ks_1']
# indices = [[0, 1, 2], [3, 4, 5]]
cellfile = f'{DATA}/h2o_1.xyz'
pplist = {'H' : f'{DATA}/H_ONCV_PBE-1.2.upf', 'O' : f'{DATA}/O_ONCV_PBE-1.2.upf'}
subkeys = ['sub_ks_0']
indices = [[0, 1, 2]]
ecut = 1200*ENERGY_CONV["eV"]["Hartree"]
cellsplit = [0.5, 0.5, 0.5]

### 2. Create the optimizer object! 

In [12]:
opt = get_optimizer(pplist,cellfile, subkeys, indices, ecut, cellsplit)

********************************************************************************
Parallel version (MPI) on        1 processors
              eDFTpy Version : 0.0.1dev0+git20260603.e66a3ba
               DFTpy Version : 2.2.1.dev23+g3053fca65
********************************************************************************
GlobalCell grid [72 72 72]
setting key: H -> ../../../DATA//H_ONCV_PBE-1.2.upf


hwloc/linux: Ignoring PCI device with non-16bit domain.
Pass --enable-32bits-pci-domain to configure to support such devices
(warning: it would break the library ABI, don't enable unless really needed).
hwloc/linux: Ignoring PCI device with non-16bit domain.
Pass --enable-32bits-pci-domain to configure to support such devices
(warning: it would break the library ABI, don't enable unless really needed).


setting key: O -> ../../../DATA//O_ONCV_PBE-1.2.upf
ncharge(sub_ks_0): 0 7.9999999999999964


In [13]:
def print_density(opt=opt):
    '''Print_density(opt)
    (1) computes total integrated charge of global density
    (2) prints only from MPI rank 0
    '''
    n=opt.gsystem.density.integral()
    if opt.gsystem.graphtopo.comm.rank==0:
        print('**Charge of global system: {}'.format(n))

In [14]:
def print_density_driver_all(opt=opt):
    ''' Print_density_driver_all(opt)
    (1) loops over all subsystem drivers
    (2) prints integrated charge per subsystem
    (3) MPI-safe (rank-filtered output)
    '''
    for isub, driver in enumerate(opt.drivers):
        if driver is None: continue
        if driver.comm.rank==0:
            n=driver.density.integral()
            print('****Charge of driver {} is : {}'.format(isub,n))

In [15]:
def print_ext_energy(opt=opt, isub=0):
    ''' Print_ext_energy(opt, isub=0)
    (1) evaluates external functional energy for a selected subsystem
    (2) prints result (rank 0 only) 
    '''
    driver=opt.drivers[isub]
    if driver is None: return
    if driver.comm.rank==0:
        energy=ext_functional(driver.density).energy
        print('******Ext energy of {} driver : {}'.format(isub,energy))

### 3. Run the Optimization !

In [16]:
opt.attach(print_density, interval=1)
opt.attach(print_density_driver_all, interval=2)
opt.attach(print_ext_energy, interval=1)
opt.optimize()

Begin optimize
Optimization options :
{'delay': 2,
 'econv': 3e-06,
 'maxiter': 800,
 'maxtime': 0,
 'ncheck': 2,
 'olevel': 2,
 'pconv': 3.0000000000000004e-08,
 'pconv_sub': array([3.e-08]),
 'sdft': 'sdft',
 'voltage': 0}
Update density : 7.999999999999997
**Charge of global system: 7.999999999999997
****Charge of driver 0 is : 7.9999999999999964
******Ext energy of 0 driver : 9.358214360532475
          Step    Energy(a.u.)            dE              dP        dC        Time(s)         
ncharge(sub_ks_0): 1 7.9999999999999964
**Charge of global system: 7.999999999999997
******Ext energy of 0 driver : 7.753937330316722
res_norm(sub_ks_0): 1  0.1537281691004988  0.012352241414988094
Norm of reidual density : 
[0.01235224]
Energy of reidual density : 
[0.79065993]
sub_energy(sub_ks_0): 1  -9.854159070276829
----------------------------------------------------------------------------------------------------
   Embed: 1       4.377154179708E+00      4.377154E+00    7.91E-01  1.24E-02  4

## 4. Get optimizated density! 

In [17]:
from dftpy.ions import Ions
from ase.io import read
from ase.visualize import view

def get_density(cellfile):
    ions = Ions.from_ase(read(cellfile))
    rho_ff = opt.density
    rho_ff.write('water.xsf', ions=ions)

In [18]:
get_density(cellfile)

In [21]:
#clearn after run
! rm -rf sub_ks_*